### Ejemplo básico

In [4]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

#crea un experimento, no tiene que existir
#mlflow.create_experiment("mi_experimento")
#tambiénm crea un experimento, puede existir
mlflow.set_experiment("mi_experimento")
#mlflow.set_experiment_tag("scikit-learn","lr")
# Tags
mlflow.set_tag("tipo", "basico")
#ejecucion
mlflow.end_run()
with mlflow.start_run(run_name="run_primera"):
    mlflow.log_param("depth", 6)
    mlflow.log_metric("auc", 0.81)


🏃 View run merciful-stoat-698 at: http://127.0.0.1:5000/#/experiments/1/runs/74ac922a319b44beb6a81dbd817efcd4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run run_primera at: http://127.0.0.1:5000/#/experiments/1/runs/f72f397fcc2d4036a1f82746603fdee9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [18]:
#Crea una carpeta en disco ./mlruns
mlflow.set_tracking_uri("file:./mlruns")  # crea/usa ./mlruns en tu proyecto


### Randomforest 

In [6]:
import os
import json
from pathlib import Path
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

import matplotlib.pyplot as plt

In [38]:
def main():
    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    mlflow.set_experiment("iris RandomForestClassifier")

    X, y = load_iris(return_X_y=True)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    mlflow.end_run()
    with mlflow.start_run(run_name="run_random"):
        params = {"n_estimators": 200, "max_depth": 5, "random_state": 42}
        mlflow.log_params(params)
        
        
        model = RandomForestClassifier(**params)
        model.fit(Xtr, ytr)
        #acc
        acc = model.score(Xte, yte)
        print(f'acc: {acc}')
        #crea una metrica accuracy
        mlflow.log_metric("accuracy", acc)
        
        # Tags
        mlflow.set_tag("dataset", "load_iris")
        #predict
        y_pred = model.predict(Xte)
        # Artefactos: matriz de confusión (png)
        cm = confusion_matrix(yte, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot()
        Path("artifacts").mkdir(exist_ok=True)
        fig_path = Path("artifacts") / "confusion_matrix_.png"
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(str(fig_path), artifact_path="figures")
    
        #guarda el modelo en mlflow
        mlflow.sklearn.log_model(model, artifact_path="model")
        #cargo el modelo
        #model = mlflow.sklearn.load_model(f"runs:/run_random/mode/model.pkl")
        print(model)


In [39]:
if __name__ == "__main__":    
    main()

2026/02/04 14:57:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


acc: 1.0
RandomForestClassifier(max_depth=5, n_estimators=200, random_state=42)
🏃 View run run_random at: http://127.0.0.1:5000/#/experiments/2/runs/7e82763ec0c1463ea4946bca2d4ef939
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### Regresión

In [15]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("iris LogisticRegression")

X, y  = load_iris(return_X_y=True)
model = LogisticRegression(max_iter=200).fit(X, y)

#guarda metricas y parametros 
mlflow.sklearn.autolog()



print(model)
with mlflow.start_run(run_name="run_regresion"):
    mlflow.log_param("max_iter", 200)
    
    # Log del modelo como artefacto
    mlflow.sklearn.log_model(model, artifact_path="model")
    #guarda el modelo en disco
    #mlflow.sklearn.save_model(model, 'c:/users/mlruns/models')
    #guardo tambien en mlflow
    #guarda el modelo en mlflow
    mlflow.sklearn.log_model(model, artifact_path="model")
    # Registrar el modelo (crea una versión en el registry)
    result = mlflow.register_model(
        model_uri=f"runs:/{mlflow.active_run().info.run_id}/model",
        name="IrisLogReg"
    )
    
    #cargo el modelo
    #model = mlflow.sklearn.load_model('c:/users/mamen/mlruns/models')
    print(model)
print("Registered model version:", result.version)


2026/02/03 18:52:57 INFO mlflow.tracking.fluent: Experiment with name 'iris LogisticRegression' does not exist. Creating a new experiment.
2026/02/03 18:52:57 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.4.0 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/02/03 18:52:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


LogisticRegression(max_iter=200)


2026/02/03 18:52:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'IrisLogReg'.
2026/02/03 18:53:00 WARNING mlflow.tracking._model_registry.fluent: Run with id 96430bca887247db803029aa03bd4a03 has no artifacts at artifact path 'model', registering model based on models:/m-842193cc9b014e10bad90de18fa84cc6 instead
2026/02/03 18:53:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IrisLogReg, version 1


LogisticRegression(max_iter=200)
🏃 View run run_regresion at: http://127.0.0.1:5000/#/experiments/3/runs/96430bca887247db803029aa03bd4a03
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Registered model version: 1


Created version '1' of model 'IrisLogReg'.


## 0.0.4 Inferir con un modelo

In [35]:
import mlflow

# 1. Seleccionar el experimento
experiment = mlflow.get_experiment_by_name("iris LogisticRegression")

# 2. Buscar el run por su nombre ("myruy")
client = mlflow.tracking.MlflowClient()

runs = client.search_runs(experiment_ids=[experiment.experiment_id],filter_string="tags.mlflow.runName = 'run_regresion'"
)

if len(runs) == 0:
    raise ValueError("No se encontró un run con nombre 'run_regresion' en el experimento 'iris LogisticRegression'.")

run_id = runs[0].info.run_id
print("Run ID encontrado:", run_id)

# 3. Construir la ruta del artefacto del modelo
model_uri = f"runs:/{run_id}/model"

# 4. Cargar el modelo
model = mlflow.pyfunc.load_model(model_uri)

print("Modelo cargado correctamente.")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Run ID encontrado: 96430bca887247db803029aa03bd4a03


Modelo cargado correctamente.


In [46]:
# Ejemplo de datos (ajusta columnas/orden a tu entrenamiento)
X_new = pd.DataFrame([
    {"sepal length (cm)": 5.9, "sepal width (cm)": 3.0, "petal length (cm)": 5.1, 
     "petal width (cm)": 1.8}
])

y_pred = model.predict(X_new)
print(y_pred)

[2]


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
